In [1]:
# Neo4j and PostgreSQL connection details
NEO4J_URI = "neo4j://182.176.180.82:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "graph2024"
NEO4J_DATABASE = "neo4j"

In [6]:
import os
import pandas as pd
from neo4j import GraphDatabase
#from psycopg2 import connect

In [4]:
# Set environment variables for Neo4j
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["NEO4J_DATABASE"] = NEO4J_DATABASE

In [7]:
# PostgreSQL connection setup
pg_conn = connect(
    dbname="abc", 
    user="postgres", 
    password="Postgres24*", 
    host="182.176.180.82", 
    port="5897"
)

In [9]:
# Function to create table nodes in Neo4j
def create_table_node(tx, table_name):
    tx.run(f"CREATE (n:Table {{name: $name}})", name=table_name)

In [11]:
# Function to establish relationships between the table nodes
def create_table_relationships(tx):
    relationships = [
        ('src1_file', 'src1_staging_data'),
        ('sr2_file1', 'src2_staging_data1'),
        ('src2_file2', 'src2_staging_data2'),
        ('src2_staging_data1', 'stg_src2_kde'),
        ('src2_staging_data2', 'stg_src2_kde'),
        ('src1_staging_data', 'stg_src1_kde'),
        ('stg_src1_kde', 'stg_src1_stitched'),
        ('stg_src2_kde', 'stg_src1_stitched'),
        ('stg_src1_stitched', 'stg_acct_txns'),
        ('stg_acct_txns', 'fin_acct_txns'),
    ]
    for start_node, end_node in relationships:
        tx.run(f"MATCH (a:Table {{name: $start}}), (b:Table {{name: $end}}) "
               "CREATE (a)-[:CONNECTED_TO]->(b)", start=start_node, end=end_node)

In [13]:
# Function to attach data from PostgreSQL to the Neo4j table node
def attach_table_data(tx, table_name, data):
    tx.run(f"""
    MATCH (n:Table {{name: $name}})
    SET n.data = $data
    """, name=table_name, data=data)
    

In [15]:
# Function to fetch a summary or sample of data from PostgreSQL for each table
def fetch_table_data(pg_conn, table_name):
    try:
        query = f"SELECT * FROM {table_name}"  # Fetch all rows
        df = pd.read_sql_query(query, pg_conn)
        data_summary = df.to_json(orient="records")  # Convert to JSON format
        return data_summary
    except Exception as e:
        return f"Error fetching data from {table_name}: {str(e)}"

In [17]:
# Clean dataset first (optional)
graph = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [19]:
with graph.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

In [21]:
# Neo4j driver setup
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [23]:
# Define table names
tables = [
    'fin_acct_txns', 'sr2_file1', 'src1_file', 'src1_staging_data', 
    'src2_file2', 'src2_staging_data1', 'src2_staging_data2', 
    'stg_acct_txns', 'stg_src1_kde', 'stg_src1_stitched', 'stg_src2_kde'
]

In [25]:
# Transfer data: create nodes, relationships, and attach data
with driver.session() as session:
    # Create table nodes
    for table in tables:
        session.write_transaction(create_table_node, table)

    # Create relationships between the table nodes
    session.write_transaction(create_table_relationships)

    # Attach PostgreSQL data to each node
    for table in tables:
        table_data = fetch_table_data(pg_conn, table)
        session.write_transaction(attach_table_data, table, table_data)

# Close connections
driver.close()
pg_conn.close()

C:\Users\amnas\AppData\Local\Temp\ipykernel_21520\465345081.py:5: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_table_node, table)
C:\Users\amnas\AppData\Local\Temp\ipykernel_21520\465345081.py:8: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_table_relationships)
C:\Users\amnas\AppData\Local\Temp\ipykernel_21520\1016483441.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, pg_conn)
C:\Users\amnas\AppData\Local\Temp\ipykernel_21520\465345081.py:13: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(attach_table_data, table, table_data)


In [11]:
# Function to visualize the graph using yFiles for Jupyter
from yfiles_jupyter_graphs import GraphWidget

In [13]:
default_cypher = """
MATCH (n:Table)-[r:CONNECTED_TO]->(m:Table)
RETURN n, r, m
"""

In [15]:
def filter_properties(node_data, required_amt=57001, required_dt="23-Dec-2024"):
    """
    Filter node properties based on tran_amt and tran_dt.
    This function parses the 'data' field, checks if 'tran_amt' and 'tran_dt' match the criteria,
    and returns filtered properties.
    """
    import json

    try:
        # Parse the 'data' field (assuming it's in JSON format)
        data_list = json.loads(node_data)  # Expecting 'node_data' to be a list of dictionaries
        
        # Ensure that node_data is a list of dictionaries
        if isinstance(data_list, list):
            # Loop through the list of data entries (dictionaries)
            for entry in data_list:
                # Check if the dictionary contains 'tran_amt' and 'tran_dt'
                if entry.get("tran_amt") == required_amt and entry.get("tran_dt") == required_dt:
                    return entry  # Return the matching dictionary
        
        return None  # Return None if no matching data is found
    except json.JSONDecodeError:
        return None  # Handle cases where the data is not valid JSON



In [17]:
def showGraph(cypher: str = default_cypher):
    # Create a neo4j session to run queries
    driver = GraphDatabase.driver(
        uri=os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])
    )
    session = driver.session()

    # Fetch the graph data
    result_graph = session.run(cypher).graph()

    # Create a widget for rendering the graph
    widget = GraphWidget(graph=result_graph)

    # Create a dictionary to store filtered data for each node
    node_filtered_data = {}
    node_name = {}

    # Filter node properties to display specific data in the properties panel
    for node in result_graph.nodes:
        node_data = node.get("data", "")
        node_name[node.id] = node.get("name", "")
        
        # Filter properties based on 'tran_amt' and 'tran_dt'
        filtered_properties = filter_properties(node_data)
        
        # Store the filtered data in the dictionary
        if filtered_properties:
            node_filtered_data[node.id] = filtered_properties  # Map node ID to filtered data
        else:
            node_filtered_data[node.id] = {"info": "No relevant data"}  # Fallback if no match

    # Color mapping function
    def custom_color_mapping(index, node):
        if 'properties' in node:
            return node['properties'].get('label', 'default_label')  # Provide a default label
        return 'default_label'  # Default case if properties are missing

    widget.node_color_mapping = custom_color_mapping

    # Node property mapping
    def custom_node_property_mapper(node):
        node_id = node.get('id')
        return node_filtered_data.get(node_id, {"info": "No relevant data", "error": "Missing node properties"})

    widget.node_property_mapping = custom_node_property_mapper

    # Node label mapping
    def custom_node_label_mapper(node):
        node_id = node.get('id')
        return node_name.get(node_id, "Unnamed Node")

    widget.node_label_mapping = custom_node_label_mapper

    return widget

# Display the graph with filtered node properties
showGraph()


C:\Users\amnas\AppData\Local\Temp\ipykernel_23336\2488032069.py:22: DeprecationWarning: `id` is deprecated, use `element_id` instead
  node_name[node.id] = node.get("name", "")
C:\Users\amnas\AppData\Local\Temp\ipykernel_23336\2488032069.py:29: DeprecationWarning: `id` is deprecated, use `element_id` instead
  node_filtered_data[node.id] = filtered_properties  # Map node ID to filtered data


GraphWidget(layout=Layout(height='610px', width='100%'))